# FDSI benchmark v1.1: apply the winning config -- pareto (min-sv and knee)

Applies the config(s) `03a_pareto_optimisation.ipynb` promoted to all 100 recordings of the benchmark with
`process_from_calib_end` (CBSS's output over the 5 s calibration window, adaptation forwards
from its end), then repeats v1.0's `../fdsi_benchmark/04a_pareto_application.ipynb` results section with the v1.0
and v1.1 configs side by side, plus both fixed baselines.

Computes whatever is missing under `OUTPUTS_ROOT/adaptation_v1_1/<sub>/lr_fixed/<sampler>/`
(`RUN_APPLY_BEST=True`); a rerun only loads.

## Config

In [ ]:
RUN_APPLY_BEST = True   # apply the promoted config(s) to every recording not yet cached

import sys
from pathlib import Path
sys.path.insert(0, '../..')          # package root
sys.path.insert(0, str(Path.cwd()))  # this folder, for fdsi_v11 (which imports ../fdsi_benchmark/fdsi_common)

import fdsi_v11 as fv
fc = fv.fc

g = fv.Grid.from_yaml(Path('../../configs/data_configs/fdsi_benchmark_grid.yaml'))
BASE_CONFIG_PATH = Path('../../configs/adapt_configs/default_muniverse.yaml')   # v1.0's base config

print(f'{len(g.recordings())} recordings; calibration window [0, {g.cal_end}) samples; '
      f'pool {g.pool_conditions} ({g.optim_sub}, {g.optim_snr} dB); holdout {g.holdout_conditions}')

BRANCHES = ['pareto_min_sv', 'pareto_knee']
CONFIGS = fv.configs(*BRANCHES)
CONFIG_ORDER = [label for label, *_ in CONFIGS]
CONFIG_ORDER

## Calibration

The cached v1.0 calibrations (`../fdsi_benchmark/01_calibration.ipynb`), unchanged: both versions
track exactly the same units, so their results can be paired unit by unit.

In [ ]:
calibrations = fv.load_calibrations(g)
print(f'Calibration available: {sum(c is not None for c in calibrations.values())}/{len(calibrations)} recordings.')

## Read back the winning configs

The v1.1 configs `03` promoted to `configs/adapt_configs/`, next to v1.0's promoted winner.

In [ ]:
from adapt_decomp.adaptation import AdaptConfig

best_configs = {b: AdaptConfig.from_yaml(fv.PROMOTED_DIR / fv.BRANCHES[b].promoted) for b in BRANCHES}
fv.hyperparameter_table(BRANCHES)

## Apply the winning configs to the full benchmark grid

In [ ]:
results = {b: fv.apply_all(g, calibrations, config, fv.BRANCHES[b].sampler, RUN_APPLY_BEST)
           for b, config in best_configs.items()}

## Results

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from adapt_decomp import AdaptationResult
from adapt_decomp.utils.plots import (plot_metric_boxplot, plot_metric_heatmap, plot_phase_bar,
                                      plot_roa_summary_figure, plot_sources)

sns.set_theme(style='whitegrid', font_scale=1.1)

### Example recording -- sources, v1.0 vs v1.1

The v1.1 baseline and every config of this notebook on one pooled recording (dashed:
calibration end, wrist angle in grey).

In [ ]:
sub, cond, snr = g.optim_sub, g.triangular[0], g.optim_snr
shown = [c for c in CONFIGS if c[0] != 'fixed (v1.0)']
example = {}
for label, version, lr_mode, sampler in shown:
    path, _ = fv.result_paths(g.adapt_dir(version), sub, cond, snr, lr_mode, sampler)
    if path.exists():
        example[label] = AdaptationResult.load(path)
angle = fc.load_angle_profile(g.data_dir, sub, cond)
ts_full = np.arange(next(iter(example.values())).spikes.shape[0]) / g.fs

axs = plot_sources(
    sources={label: out.sources.numpy() for label, out in example.items()},
    timestamps=ts_full,
    spikes={label: out.spikes.numpy() for label, out in example.items()},
    time_range=(0, ts_full[-1]),
    square_sources=True,
)
for ax in axs:
    ax.axvline(g.cal_end / g.fs, color='k', ls='--', lw=1)
    ax_angle = ax.twinx()
    ax_angle.plot(ts_full, angle, color='grey', alpha=0.3, zorder=0)
    ax_angle.set_yticks([])
axs[0].set_title(f'Sources -- {sub} {cond} snr{snr}dB')
plt.show()

### Heatmaps -- mean / median / % good units RoA, and SIL

In [ ]:
df_roa = fv.roa_table(g, CONFIGS)
configs_present = [c for c in CONFIG_ORDER if c in set(df_roa['config'])]
print(f'{len(df_roa)} (recording, unit, config) rows; configs present: {configs_present}')

In [ ]:
plot_metric_heatmap(df_roa, 'roa_pct', configs_present, g.conditions, g.snr_levels, agg='mean', vmin=0, vmax=100)
plt.suptitle('FDSI benchmark -- mean RoA by condition x SNR', fontsize=12)
plt.show()

In [ ]:
plot_metric_heatmap(df_roa, 'roa_pct', configs_present, g.conditions, g.snr_levels, agg='median', vmin=0, vmax=100)
plt.suptitle('FDSI benchmark -- median RoA by condition x SNR', fontsize=12)
plt.show()

In [ ]:
plot_metric_heatmap(df_roa, 'roa_pct', configs_present, g.conditions, g.snr_levels,
                    agg='pct_ge_threshold', threshold=90.0, vmin=0, vmax=100)
plt.suptitle('FDSI benchmark -- % units with RoA >= 90%, by condition x SNR', fontsize=12)
plt.show()

In [ ]:
df_sil = fv.sil_table(g, CONFIGS)
plot_metric_heatmap(df_sil, 'sil', configs_present, g.conditions, g.snr_levels, agg='mean', vmin=0, vmax=1)
plt.suptitle('FDSI benchmark -- mean full-recording SIL by condition x SNR -- GT-free RoA proxy', fontsize=12)
plt.show()

### Boxplots -- per-unit RoA and SIL by condition, hue=SNR

In [ ]:
axs = plot_metric_boxplot(df_roa, 'roa_pct', configs_present, g.conditions, g.snr_levels, vmin=0, vmax=105)
axs[0].set(ylabel='RoA (%)')
plt.show()

In [ ]:
axs = plot_metric_boxplot(df_sil, 'sil', configs_present, g.conditions, g.snr_levels, vmin=0.7, vmax=1)
axs[0].set(ylabel='SIL')
plt.show()

### Paired per-unit change, v1.1 vs v1.0

Same calibrations, same units: each unit's RoA under the v1.1 config minus under its v1.0
counterpart (the fixed baselines too), over the full recording and after the calibration
window only.

In [ ]:
import pandas as pd

df_post = fv.window_roa_table(g, CONFIGS, slice(g.cal_end, None))
pd.concat({'full recording': fv.paired_summary(df_roa, fv.pairs(*BRANCHES)),
           'after calibration': fv.paired_summary(df_post, fv.pairs(*BRANCHES))}).round(3)

### Phase RoA and pool vs holdout summary -- triangular contractions

In [ ]:
df_phase = fv.phase_table(g, CONFIGS)
plot_phase_bar(df_phase, configs_present, g.triangular, g.snr_levels, fc.PHASE_ORDER, fc.PHASE_LABELS)
plt.suptitle('Phase RoA -- triangular contractions', fontsize=13)
plt.show()

In [ ]:
adapted_labels = [c for c in configs_present if not c.startswith('fixed')]
plot_roa_summary_figure(df_roa, df_phase, adapted_labels, g.conditions, g.holdout_conditions,
                        g.snr_levels, fc.PHASE_ORDER, fc.PHASE_LABELS)
plt.suptitle('FDSI benchmark -- pool (blue x-labels) vs holdout (red bold x-labels)', fontsize=12)
plt.show()